In [2]:
import numpy as np
import unicodedata
import os
import re
import xml.etree.ElementTree as ET
from music21 import stream, note, chord, meter, tempo, duration, instrument, clef, expressions

In [3]:
"""Funciones"""

def get_time_signature_from_offsets(arr):
    # Filtrar compás 1
    compas1 = arr[arr[:, 4] == 1]
    if compas1.size == 0:
        raise ValueError("No hay notas en el compás 1 para determinar el compás.")
    max_offset = np.max(compas1[:, 1])  # columna de offset
    numerador = int(round(max_offset))
    return f"{numerador}/4"

# Normalizador de texto: sin acentos, minúsculas, sin puntuación ni espacios extra
def normalize(text):
    text = text.lower()
    text = ''.join(c for c in unicodedata.normalize('NFD', text)
                   if unicodedata.category(c) != 'Mn')  # elimina acentos
    text = re.sub(r'[^a-z0-9]+', '', text)  # elimina puntuación y espacios
    return text
def get_tempo(term):
    norm = normalize(term)
    return tempo_dict.get(norm, None)

def midi_to_note(midi_val):
    return note.Note(midi_val) if midi_val >= 0 else note.Rest()

def array_to_voice(arr, time_signature):
    part = stream.Part()
    clef_to_use = choose_clef(arr[:, 3])
    part.append(clef_to_use)
    part.append(meter.TimeSignature(time_signature))

        
    current_measure_number = -1
    measure = None

    for measure_number in sorted(set(arr[:, -1])):
        measure = stream.Measure(number=int(measure_number))
        current_measure_number = measure_number

        # Extraer eventos de este compás
        notes_in_measure = arr[arr[:, -1] == measure_number]

        # Agrupar por onset dentro del compás
        onsets = sorted(set(notes_in_measure[:, 0]))
        for onset in onsets:
            group = notes_in_measure[notes_in_measure[:, 0] == onset]
            pitches = group[:, 3]
            durations = group[:, 2]

            if all(p == -1 for p in pitches):  # Todo es silencio
                r = note.Rest()
                r.duration = duration.Duration(durations[0])
                r.offset = onset
                measure.insert(onset, r)
            elif sum(p != -1 for p in pitches) == 1:
                idx = pitches != -1
                n = note.Note(int(pitches[idx][0]))
                n.duration = duration.Duration(durations[idx][0])
                n.offset = onset
                measure.insert(onset, n)
            else:
                chord_pitches = [int(p) for p in pitches if p != -1]
                dur_val = durations[pitches != -1][0]
                c = chord.Chord(chord_pitches)
                c.duration = duration.Duration(dur_val)
                c.offset = onset
                measure.insert(onset, c)

        part.append(measure)

    return part



def choose_clef(midi_vals):
    notes = [v for v in midi_vals if v >= 0]
    if not notes:
        return clef.TrebleClef()  # por defecto
    min_pitch = min(notes)
    max_pitch = max(notes)
    avg_pitch = sum(notes) / len(notes)

    # Algunas reglas simples (puedes ajustarlas a tus necesidades)
    if avg_pitch < 59:
        return clef.BassClef()
    elif avg_pitch > 62:
        return clef.TrebleClef()
    else:
        return clef.AltoClef()
    
def insertar_tempo_en_musicxml(path_in, path_out, tempo_bpm):
    tree = ET.parse(path_in)
    root = tree.getroot()

    # Namespaces de MusicXML
    ns = {'': 'http://www.musicxml.org/ns/musicxml'}

    # Buscar primer compás (measure) del primer part
    first_part = root.find('part')
    first_measure = first_part.find('measure')

    # Crear bloque <direction>
    direction = ET.Element('direction', attrib={'placement': 'above'})
    direction_type = ET.SubElement(direction, 'direction-type')
    metronome = ET.SubElement(direction_type, 'metronome')
    beat_unit = ET.SubElement(metronome, 'beat-unit')
    beat_unit.text = 'quarter'
    per_minute = ET.SubElement(metronome, 'per-minute')
    per_minute.text = str(tempo_bpm)

    sound = ET.SubElement(direction, 'sound', attrib={'tempo': str(tempo_bpm)})

    # Insertar al inicio del primer compás
    first_measure.insert(1, direction)

    # Guardar nuevo archivo
    tree.write(path_out, encoding='utf-8', xml_declaration=True)

In [4]:
float(np.random.choice(np.arange(60,72)))

67.0

In [ ]:
array = np.array([
        [0.0, 1.0, 1.0, 60.0, 1.0],
        [1.0, 2.0, 1.0, 64.0, 1.0],
        [2.0, 3.0, 1.0, 67.0, 1.0],
        [3.0, 4.0, 1.0, 64.0, 1.0],
        [0.0, 1.0, 1.0, 60.0, 2.0],
        [1.0, 2.0, 1.0, 64.0, 2.0],
        [2.0, 3.0, 1.0, 67.0, 2.0],
        [3.0, 4.0, 1.0, 64.0, 2.0],
        [0.0, 1.0, 1.0, 60.0, 3.0],
        [1.0, 2.0, 1.0, 64.0, 3.0],
        [2.0, 3.0, 1.0, 67.0, 3.0],
        [3.0, 4.0, 1.0, 64.0, 3.0],
        [0.0, 1.0, 1.0, 60.0, 4.0],
        [1.0, 2.0, 1.0, 64.0, 4.0],
        [2.0, 3.0, 1.0, 67.0, 4.0],
        [3.0, 4.0, 1.0, 64.0, 4.0]])

array = np.array([
        [0.0, 1.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 1.0],
        [1.0, 2.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 1.0],
        [2.0, 3.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 1.0],
        [3.0, 4.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 1.0],
        [0.0, 1.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 2.0],
        [1.0, 2.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 2.0],
        [2.0, 3.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 2.0],
        [3.0, 4.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 2.0],
        [0.0, 1.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 3.0],
        [1.0, 2.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 3.0],
        [2.0, 3.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 3.0],
        [3.0, 4.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 3.0],
        [0.0, 1.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 4.0],
        [1.0, 2.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 4.0],
        [2.0, 3.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 4.0],
        [3.0, 4.0, 1.0, np.random.choice([60.0,62.,64.,65.,67.,69.,71.]), 4.0]])

array = np.array([
        [0.0, 1.0, 1.0, float(np.random.choice(np.arange(60,72))), 1.0],
        [1.0, 2.0, 1.0, float(np.random.choice(np.arange(60,72))), 1.0],
        [2.0, 3.0, 1.0, float(np.random.choice(np.arange(60,72))), 1.0],
        [3.0, 4.0, 1.0, float(np.random.choice(np.arange(60,72))), 1.0],
        [0.0, 1.0, 1.0, float(np.random.choice(np.arange(60,72))), 2.0],
        [1.0, 2.0, 1.0, float(np.random.choice(np.arange(60,72))), 2.0],
        [2.0, 3.0, 1.0, float(np.random.choice(np.arange(60,72))), 2.0],
        [3.0, 4.0, 1.0, float(np.random.choice(np.arange(60,72))), 2.0],
        [0.0, 1.0, 1.0, float(np.random.choice(np.arange(60,72))), 3.0],
        [1.0, 2.0, 1.0, float(np.random.choice(np.arange(60,72))), 3.0],
        [2.0, 3.0, 1.0, float(np.random.choice(np.arange(60,72))), 3.0],
        [3.0, 4.0, 1.0, float(np.random.choice(np.arange(60,72))), 3.0],
        [0.0, 1.0, 1.0, float(np.random.choice(np.arange(60,72))), 4.0],
        [1.0, 2.0, 1.0, float(np.random.choice(np.arange(60,72))), 4.0],
        [2.0, 3.0, 1.0, float(np.random.choice(np.arange(60,72))), 4.0],
        [3.0, 4.0, 1.0, float(np.random.choice(np.arange(60,72))), 4.0]])

array = np.array([
        [0.0, 1.0, 1.0, 60.0, 1.0],
        [1.0, 2.0, 1.0, 62.0, 1.0],
        [2.0, 3.0, 1.0, 64.0, 1.0],
        [3.0, 4.0, 1.0, 65.0, 1.0],
        [0.0, 1.0, 1.0, 67.0, 2.0],
        [1.0, 2.0, 1.0, 69.0, 2.0],
        [2.0, 3.0, 1.0, 71.0, 2.0],
        [3.0, 4.0, 1.0, 72.0, 2.0],
        [0.0, 1.0, 1.0, 74.0, 3.0],
        [1.0, 2.0, 1.0, 76.0, 3.0],
        [2.0, 3.0, 1.0, 77.0, 3.0],
        [3.0, 4.0, 1.0, 79.0, 3.0],
        [0.0, 1.0, 1.0, 81.0, 4.0],
        [1.0, 2.0, 1.0, 83.0, 4.0],
        [2.0, 3.0, 1.0, 84.0, 4.0],
        [3.0, 4.0, 1.0, 86.0, 4.0]])

array_complete = np.array((array,))
np.shape(array_complete)

(1, 16, 5)

In [6]:
"""Generador de array"""
score = stream.Score() 
for i in range(np.shape(array_complete)[0]):
    arr = array_complete[i,:,:]
    print(arr)
    print(np.shape(arr))
    p = array_to_voice(arr, get_time_signature_from_offsets(arr))
    instrumento = "Piano"
    p.insert(0, instrument.fromString(instrumento))
    score.append(p)
score.write('musicxml', fp='partitura_generada.xml')
insertar_tempo_en_musicxml('partitura_generada.xml', 'partitura_escala_180.xml', 180)

[[ 0.  1.  1. 60.  1.]
 [ 1.  2.  1. 62.  1.]
 [ 2.  3.  1. 64.  1.]
 [ 3.  4.  1. 65.  1.]
 [ 0.  1.  1. 67.  2.]
 [ 1.  2.  1. 69.  2.]
 [ 2.  3.  1. 71.  2.]
 [ 3.  4.  1. 72.  2.]
 [ 0.  1.  1. 74.  3.]
 [ 1.  2.  1. 76.  3.]
 [ 2.  3.  1. 77.  3.]
 [ 3.  4.  1. 79.  3.]
 [ 0.  1.  1. 81.  4.]
 [ 1.  2.  1. 83.  4.]
 [ 2.  3.  1. 84.  4.]
 [ 3.  4.  1. 86.  4.]]
(16, 5)


In [6]:
path = r'data\humdrum-data-numpy\beethoven\piano\sonata\sonata14-3'
score = stream.Score() 
for filename in sorted(os.listdir(path)):
    if filename.endswith(".npy"):
        arr = np.load(f"{path}/{filename}")
        p = array_to_voice(arr, get_time_signature_from_offsets(arr), 180)
        instrumento = f"{filename[:-11]}".replace("_", " ")
        instrumento = "Piano"
        if instrumento == "Keyboard":
            instrumento = "Piano"
        if instrumento == "StringInstrument":
            instrumento = "Strings"
        if instrumento == "Brass":
            instrumento = "Trumpet" 
        p.insert(0, instrument.fromString(instrumento))
        score.append(p)
score.write('musicxml', fp='partitura_generada.xml')

WindowsPath('d:/La formula secreta de la cangreburger/Documentos/uaem/octavo semestre/Tesis/Ritmos/partitura_generada.xml')

In [34]:
instrument.getAllNamesForInstrument(instrument.KeyboardInstrument())


{'english': [],
 'french': [],
 'german': [],
 'italian': [],
 'russian': [],
 'spanish': [],
 'abbreviation': []}

In [ ]:
_raw_tempo_dict  = {
    # Francés
    "Très lent": 40,
    "Lent": 50,
    "Assez lent": 60,
    "Modéré": 90,
    "Très modéré": 80,
    "Allant": 100,
    "Animé": 120,
    "Très animé": 140,
    "Vif": 120,
    "Très vif": 140,
    "Rapide": 160,
    "Très rapide": 180,

    # Italiano
    "Grave": 35,
    "Largo": 40,
    "Lento": 50,
    "Larghetto": 60,
    "Adagio": 66,
    "Adagietto": 72,
    "Andante": 76,
    "Andantino": 80,
    "Marcia moderato": 83,
    "Andante moderato": 92,
    "Moderato": 96,
    "Allegretto": 104,
    "Allegro moderato": 108,
    "Allegro": 120,
    "Vivace": 140,
    "Allegrissimo": 156,
    "Presto": 168,
    "Prestissimo": 200,

    # Con modificadores (italiano)
    "Molto adagio": 60,
    "Molto allegro": 144,
    "Allegro assai": 150,
    "Allegro con brio": 160,
    "Allegro energico": 160,
    "Allegro vivace": 144,
    "Allegro molto": 150,
    "Vivacissimo": 160,
    "Più mosso": 130,  # relativo
    "Meno mosso": 90,  # relativo
    "Tempo giusto": 100,  # literal: tempo justo
    "Tempo di valse": 90,
    "Tempo di marcia": 120,
    "Tempo di minuetto": 80,
    "Tempo di gavotta": 90,
    "Tempo di polacca": 120,
    "Tempo di mazurka": 120,
    "Tempo rubato": 0,  # no tiene valor fijo
    "A tempo": 0,       # indica retorno al tempo anterior

    # Alemán
    "Sehr langsam": 40,
    "Langsam": 50,
    "Ruhig": 60,
    "Ziemlich langsam": 60,
    "Getragen": 66,  # sostenido, solemne
    "Mäßig langsam": 72,
    "Mäßig": 96,
    "Mäßig bewegt": 104,
    "Bewegt": 120,
    "Schnell": 160,
    "Sehr schnell": 180,
    "Geschwind": 160,
    "Rasch": 140,
    "Lebhaft": 138,  # vivaz

    # Inglés (moderno)
    "Slow": 60,
    "Moderate": 96,
    "Fast": 120,
    "Quickly": 132,
    "Very fast": 160,
    "With energy": 120,
    "With motion": 100,
    "Brightly": 132,
    "Steady": 96,
    "Freely": 0,  # libre
    "Calmly": 70,
    "Gently": 80,
    "Tenderly": 76,
}

tempo_dict = {normalize(k): v for k, v in _raw_tempo_dict.items()}

# Ejemplo de uso
print(get_tempo("trèS lent")) 

40
